# Lab 15: Bias-Variance — Diagnosing a Model Chosen on Its Own Score
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 110 min Core + 20 min Extension

---

**Learning Objectives:**
- Measure the optimism in a cross-validation score that was used to choose the model, against a known truth
- Say what nested cross-validation estimates, and what it cannot fix
- Split expected error into bias squared, variance and irreducible noise, and recognise the noise floor
- Write a k-fold cross-validation function for any scikit-learn model, and match the lab's numbers to every digit

**Dataset:** Simulated nonlinear data with a known truth

**Format:** Each diagnosis shows you a number an analyst would report, then what it hides, and asks you to explain the gap. Write your answers in the text cell under each set of questions.

**Deliverable:** your written diagnoses (Diagnoses 1, 2 and 3), the required `kfold_mse()` function, and `src/model_selection.py`, a finished module you save and upload.

**Time:** Core (110 min, including the required Write it yourself) + Extension (optional, 20 min)

With the 110 minutes of Core, setup (5 min), the README (10 min) and the GitHub submission (10 min), plan on about 2 hours 15 minutes, most of it after class. The optional Extension adds about 20 minutes.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Write it yourself (required, after Diagnosis 3):** `kfold_mse()`, the k-fold cross-validation error of any scikit-learn model on a table with named columns. About eight lines, from an empty cell. A check cell tells you when it is right. It is required.
> - **Extension (optional):** the one-standard-error rule, from the task.
>
> **You write, in text cells:** two answers in each of Diagnoses 1, 2 and 3.
>
> **You run and read:** everything else. That is the analyst's search and the 20,000-point reveal, the search-size table, the nested cross-validation in Diagnosis 2, the bias-variance simulation in Diagnosis 3, and Ship It's finished module, `model_selection.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints, and Write it yourself generalises Diagnosis 1's `cv_error`.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries, the seed, and a folder for Ship It's module
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

Path("src").mkdir(exist_ok=True)     # Ship It writes its module into this folder

RANDOM_STATE = 42     # the seed for the 80 training points and for every fold split
print("Setup complete.")

## Diagnosis 1: The Score That Was Selected On (30 min)

An analyst has 80 observations and twelve candidate models, polynomials of degree 1 to 12. She
cross-validates each one, keeps the degree with the smallest CV error, and reports that error as
the error to expect on new data. That number is biased, and the direction of the bias is
predictable.

The data are simulated, so the truth is known: y is the curve sin(1.4x) + 0.25x plus noise with a
standard deviation of 0.45, and x runs from 0 to 6. That lets this lab measure the bias rather
than argue about it.

**Reading the code.** `sample(n, seed)` draws n points: `r.uniform(0, 6, n)` spreads x evenly at
random between 0 and 6, and `r.normal(0, 0.45, n)` adds the noise. The same seed always gives the
same points. `np.polyfit(x, y, deg)` fits a polynomial of degree `deg` by least squares and returns
its coefficients; `np.polyval(coefs, x)` evaluates that polynomial at each x.
`KFold(n_splits=k, shuffle=True, random_state=seed)` deals the points into k folds after shuffling
them with a fixed seed, and `folds.split(x)` gives, for each fold in turn, two arrays of positions:
the points to fit on and the points held out. `x[train]` picks the points at those positions.
`cv_error` averages the k held-out mean squared errors.

In [ ]:
# GIVEN — run as-is
# A curved truth observed with noise, and the k-fold CV error of a polynomial
def truth(x):
    """The true curve, which a real analyst never sees."""
    return np.sin(1.4 * x) + 0.25 * x


def sample(n, seed):
    """n points: x between 0 and 6, y the true curve plus noise with sd 0.45."""
    r = np.random.default_rng(seed)
    x = r.uniform(0, 6, n)
    return x, truth(x) + r.normal(0, 0.45, n)


def cv_error(x, y, deg, k=5, seed=RANDOM_STATE):
    """The k-fold cross-validation mean squared error of a polynomial of degree deg."""
    folds = KFold(n_splits=k, shuffle=True, random_state=seed)
    errors = []
    for train, test in folds.split(x):
        coefs = np.polyfit(x[train], y[train], deg)                          # fit on k - 1 folds
        errors.append(np.mean((np.polyval(coefs, x[test]) - y[test]) ** 2))  # score the fold left out
    return np.mean(errors)


x_tr, y_tr = sample(80, RANDOM_STATE)     # the analyst's 80 observations

**Reading the code.** `range(1, 13)` is the whole numbers from 1 up to 12: it stops before 13.
`[cv_error(x_tr, y_tr, d) for d in degs]` builds a list with one CV error
per degree. `np.argmin(cvs)` is the *position* of the smallest one; positions count from 0 and
degrees from 1, so the degree is the position plus 1. `zip(degs, cvs)` pairs each degree with its
error, and `{d:>2}` right-aligns `d` in two characters.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Cross-validate degrees 1 to 12, keep the smallest CV error, and report it
degs = range(1, 13)
cvs = [cv_error(x_tr, y_tr, d) for d in degs]
chosen = int(np.argmin(cvs)) + 1

for d, cv in zip(degs, cvs):
    print(f"degree {d:>2}   CV error {cv:.4f}")
print(f"\nchosen degree            : {chosen}")
print(f"reported CV error        : {min(cvs):.4f}")

The next cell is the reveal. A simulation can do what the analyst cannot: draw 20,000 new points
from the same world, which stand in for the population, and score the chosen fit on them.
`20_000` is 20000; the underscore only helps you read it.

In [ ]:
# GIVEN — run as-is
# The reveal: the chosen fit's error on 20,000 new points
x_big, y_big = sample(20_000, 12345)
true_err = np.mean((np.polyval(np.polyfit(x_tr, y_tr, chosen), x_big) - y_big) ** 2)
print(f"true error on 20,000 new : {true_err:.4f}")
print(f"optimism                 : {true_err - min(cvs):+.4f}")

**Expected output:** CV errors from 0.6607 at degree 1 down to 0.1853 at degree 5, then between
0.1894 and 0.2322 for degrees 6 to 12. The analyst chooses degree 5 and reports 0.1853. On the
20,000 new points that fit's error is 0.2330, so the optimism is +0.0477.

**Does a bigger search buy more optimism?** The next cell reruns the analyst's search with only the
first 2, 4, 8 and then all 12 degrees on the menu. For each search it prints the smallest CV
error, the true error of the degree it picks, and the gap. `list(degs)[:n_cand]` is the first
`n_cand` degrees, and `{te:>10.4f}` right-aligns `te` in 10 characters with 4 decimals.

In [ ]:
# GIVEN — run as-is
# The same search with 2, 4, 8 and 12 candidates on the menu
print(f"{'candidates searched':>21}{'reported':>11}{'true':>10}{'optimism':>11}")
print("-" * 54)
for n_cand in [2, 4, 8, 12]:
    menu = list(degs)[:n_cand]
    scores = [cv_error(x_tr, y_tr, d) for d in menu]
    pick = menu[int(np.argmin(scores))]
    te = np.mean((np.polyval(np.polyfit(x_tr, y_tr, pick), x_big) - y_big) ** 2)
    print(f"{n_cand:>21}{min(scores):>11.4f}{te:>10.4f}{te - min(scores):>+11.4f}")

**Expected output:** four rows. The reported minimum reads 0.4998, 0.2144, 0.1853 and 0.1853, the
true error 0.4443, 0.2376, 0.2330 and 0.2330, and the optimism −0.0555, +0.0232, +0.0477 and
+0.0477. The rows for 8 and 12 candidates are identical because both searches pick degree 5.

This table is one training sample. Run on 200 training samples of 80 points instead (seeds 0 to
199), the average optimism is −0.012, −0.003, +0.021 and +0.024 for 2, 4, 8 and 12 candidates. Sample
by sample it is less tidy: the optimism never falls from one row to the next in 81 of the 200
samples, and rises at every step in only 3.

### YOUR DIAGNOSIS

1. The reported score is better than the truth. Name the mechanism and say why it is the same phenomenon as the winner's curse. Would the CV error of a degree the analyst chose *in advance*, before seeing any CV error, have the same problem?
2. On average over 200 samples the optimism grows with the number of candidates searched, yet in this sample the two-candidate search came out pessimistic. Explain both.

*Your diagnosis here:*

---

## Diagnosis 2: Nested Cross-Validation (20 min)

Nested cross-validation puts the analyst's whole search inside a second, outer loop of folds:

- the **outer** loop holds out one fold of 16 points;
- the **inner** loop runs the analyst's whole search, `cv_error` at every degree, on the other 64
  points only, and picks a winner;
- the winner is refitted on those 64 points and scored on the 16 held out.

The average of the five outer scores is the estimate.

The function below is written for you. Read it against the analyst's cell: the inner search is the
same list of CV errors, computed on `x_in` and `y_in` instead of `x_tr` and `y_tr`, and `inner` is
passed to `cv_error` as its number of folds `k`.

In [ ]:
# GIVEN — run as-is
# Nested CV: the analyst's degree search, run inside each of five outer folds
def nested_cv(x, y, degrees, outer=5, inner=5, seed=RANDOM_STATE):
    """Nested cross-validation mean squared error of a search over degrees."""
    outer_folds = KFold(n_splits=outer, shuffle=True, random_state=seed)
    scores = []
    for train, test in outer_folds.split(x):
        x_in, y_in = x[train], y[train]       # the search sees only these points
        inner_cvs = [cv_error(x_in, y_in, d, k=inner, seed=seed) for d in degrees]
        best = degrees[int(np.argmin(inner_cvs))]
        coefs = np.polyfit(x_in, y_in, best)  # refit the winner on all the inner points
        scores.append(np.mean((np.polyval(coefs, x[test]) - y[test]) ** 2))
    return np.mean(scores)


nested = nested_cv(x_tr, y_tr, list(degs))
print(f"reported CV minimum : {min(cvs):.4f}")
print(f"nested CV           : {nested:.4f}")
print(f"true error          : {true_err:.4f}")

**Expected output:** the reported CV minimum is 0.1853, nested CV gives 0.1910, and the true error
is 0.2330. The nested estimate depends on how the folds fall: with fold seeds 0 to 29 in place of
42 it averages 0.200, with a standard deviation of 0.016.

A simulation can also measure something no analyst can: how noisy these particular 80 points are.
Each y minus the true curve is that point's noise; the next cell squares and averages it.

In [ ]:
# GIVEN — run as-is
# The noise in the analyst's 80 points, which only a simulation can see
print(f"mean squared noise in these 80 points: {np.mean((y_tr - truth(x_tr)) ** 2):.4f}   "
      f"(in the world they come from: {0.45 ** 2:.4f})")

**Expected output:** 0.1792 in these 80 points, against 0.2025 in the world they come from.

### YOUR DIAGNOSIS

1. Nested CV costs `outer × inner` fits for every degree, and each outer fold may choose a different degree. What do you buy for those fits (it is not a better model)? Explain why a score for the *procedure*, rather than for one model, is the right number to report.
2. Nested CV gave 0.1910 against a true 0.2330. Which part of that gap can nested CV fix, and which part can't it?

*Your diagnosis here:*

---

## Diagnosis 3: Bias, Variance and the Noise Floor (15 min)

Diagnosis 1's true error has three parts, and a simulation can measure each one, because it can
draw as many training samples as it likes. For each degree, the next cell draws 200 fresh samples
of 80 points (seeds 5000 to 5199), fits each one, and records its predictions at 61 evenly spaced
x values from 0 to 6, `x_eval`. That fills a table `preds` with one row per sample and one column
per x value.

- **bias^2** averages the 200 predictions at each x, compares that average with the truth, squares
  the gap, and averages over the x values.
- **variance** is the variance of the 200 predictions at each x, averaged over the x values.
- **noise** is 0.45² = 0.2025, the variance of the noise itself, which no model can remove.

Their sum is the expected error of that degree on a new point. `axis=0` means "down the rows", so
`preds.mean(axis=0)` gives one mean per x value. `np.empty((REPS, len(x_eval)))` makes an empty
table of that shape, and `preds[r] = ...` fills row r. `preds.var(axis=0)` divides by 200, not
199.

In [ ]:
# GIVEN — run as-is
# Bias squared, variance and noise at six degrees, from 200 fresh samples of 80 points
x_eval = np.linspace(0, 6, 61)       # 61 evenly spaced x values from 0 to 6
REPS = 200
NOISE = 0.45 ** 2

print(f"{'degree':>8}{'bias^2':>10}{'variance':>10}{'noise':>8}{'expected error':>16}")
print("-" * 52)
for deg in [1, 3, 5, 7, 9, 12]:
    preds = np.empty((REPS, len(x_eval)))
    for r in range(REPS):
        xs, ys = sample(80, 5000 + r)
        preds[r] = np.polyval(np.polyfit(xs, ys, deg), x_eval)
    bias2 = np.mean((preds.mean(axis=0) - truth(x_eval)) ** 2)
    var = np.mean(preds.var(axis=0))
    print(f"{deg:>8}{bias2:>10.4f}{var:>10.4f}{NOISE:>8.4f}{bias2 + var + NOISE:>16.4f}")

**Expected output:** six rows. bias^2 falls from 0.4883 at degree 1 to 0.0068 at degree 5
and stays below 0.002 after it. Variance stays between 0.018 and 0.031 up to degree 7, then rises
to 0.0680 at degree 9 and 0.8435 at degree 12; most of degree 12's variance comes from the two
ends of the range, near 0 and 6, where a high-degree polynomial swings most. Of the six degrees in the table,
the expected error is smallest at degree 5, 0.2291, close to the 0.2330 of Diagnosis 1's fit, and the noise, 0.2025, is
most of it.

### YOUR DIAGNOSIS

1. The noise floor is 0.2025 by construction, yet Diagnosis 1's reported CV error was 0.1853. What does an estimated error below the floor tell you about that estimate?
2. On real data the truth, and so the floor, is unknown. How would you recognise that you had reached it, and why is chasing error below it a symptom rather than a goal?

*Your diagnosis here:*

---

## Write it yourself (required — 35 min)

This is the one function in this lab you write from an empty cell, and it is required. No AI:
write it yourself.

`cv_error` works only for a polynomial in one variable, held in two arrays. From Lab 16 on, the
models come from scikit-learn and the data sit in tables with named columns. Write the general
version.

**The task.** Write `kfold_mse(model, df, features, outcome, k)`. `model` is an unfitted
scikit-learn model such as `LinearRegression()`, `df` a DataFrame, `features` a list of column
names, `outcome` one column name, and `k` the number of folds. The function deals the rows of `df`
into `k` folds the way `cv_error` does (shuffled, with `random_state=RANDOM_STATE`). For each fold
it fits `model` on the other rows and computes the mean squared error of its predictions for the
fold's own rows. It returns the average of the `k` fold errors as one number: a mean squared
error, not its square root, and unrounded.

**Reading the cell below.** It builds the table the check uses: Diagnosis 1's 80 points, with y
and the powers x, x², ..., x⁵ as columns named `x1` to `x5`. A linear regression on those five
columns is a polynomial of degree 5. `pd.DataFrame({...})` builds a table from named columns, and
`df[powers]` keeps the columns named in the list `powers`. `model.fit(X, y)` fits a scikit-learn
model, X first and then y (the reverse of `sm.OLS(y, X)`), and `model.predict(X)` gives its
predictions for the rows of X. For your function: `folds.split(df)` gives positions, as it does for
an array, and `.iloc[positions]` picks the rows of a DataFrame or of a column at those positions.

In [ ]:
# GIVEN — run as-is
# Diagnosis 1's 80 points as a table: y, and x to the powers 1 to 5
poly5 = pd.DataFrame({"x1": x_tr, "x2": x_tr ** 2, "x3": x_tr ** 3,
                      "x4": x_tr ** 4, "x5": x_tr ** 5, "y": y_tr})
powers = ["x1", "x2", "x3", "x4", "x5"]

ols = LinearRegression()                    # a model with no data yet
ols.fit(poly5[powers], poly5["y"])          # fitted on all 80 rows
fitted = ols.predict(poly5[powers])         # its predictions for those same 80 rows
print(f"degree 5, fitted and scored on all 80 points: MSE {np.mean((fitted - poly5['y']) ** 2):.4f}")

**Expected output:** an MSE of 0.1644. It is a training error, scored on the same 80 points the
model was fitted to; Diagnosis 1 printed a CV error of 0.1853 for degree 5.

You need about eight lines. The check runs `kfold_mse(LinearRegression(), poly5, powers, "y", 5)`,
which should give Diagnosis 1's 0.1853 for degree 5 to every digit. It then tries your function
with 10 folds, with a different model, and on a new sample whose columns have other names. Until
you write the cell, the check says "Not written yet"; once you do, it fails with a message until
your function is right.

In [ ]:
# WRITE IT YOURSELF — required: kfold_mse(model, df, features, outcome, k)

In [ ]:
# CHECK — for kfold_mse above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "kfold_mse" not in globals():
    print("Not written yet: write kfold_mse() in the cell above and run it, then run this check again.")
else:
    import copy
    import inspect
    from sklearn.dummy import DummyRegressor

    def names_used(code):
        """Every global name a function's code refers to, including inside functions it defines."""
        names = set(code.co_names)
        for const in code.co_consts:
            if hasattr(const, "co_names"):
                names |= names_used(const)
        return names

    def same(a, b):
        """Equal to about eight significant digits: only rounding inside the computer passes."""
        return bool(np.isclose(a, b, rtol=1e-8, atol=0))

    def run(where, model, df, features, outcome, k):
        """Call kfold_mse on copies of its inputs; return one number, or stop with a plain message."""
        try:
            got = kfold_mse(copy.deepcopy(model), df.copy(), list(features), outcome, k)
        except KeyError as err:
            text = " ".join(str(err).replace("\\n", " ").split())    # the error on one line
            shown = text if len(text) <= 70 else text[:66] + " ..."
            if "[columns]" in text and any(f"'{name}'" in text for name in features):
                hint = (" This table has no such columns. Does the function use its df argument, "
                        "or poly5?")
            elif "[columns]" in text and "'x1'" in text:
                hint = (" Does the function use its features argument, or poly5's column names?")
            elif "[columns]" in text:
                hint = (" df[positions] looks for columns with those names. KFold gives row "
                        "positions: pick the rows with .iloc.")
            elif "index" in text:
                hint = (" .loc picks rows by their index labels, and this table's labels are not "
                        "0, 1, 2, .... KFold gives row positions: pick the rows with .iloc.")
            elif err.args and err.args[0] not in df.columns:
                hint = (f" This table has no column '{err.args[0]}'. Does the function use its "
                        "outcome argument, or poly5's column name?")
            else:
                hint = ""
            raise AssertionError(f"{where}, kfold_mse stopped with KeyError {shown}.{hint}") from None
        except Exception as err:
            text = " ".join(str(err).split())
            if "Data must be 1-dimensional" in text or "coerce to Series" in text:
                hint = (" df[[outcome]], with double brackets, is a one-column table, so the "
                        "predictions come back as a table too. Use df[outcome], one column.")
            elif "Expected a 2-dimensional container" in text:
                hint = (" model.fit takes the features first and the outcome second: "
                        "model.fit(X, y), the reverse of sm.OLS(y, X).")
            elif "shuffle is False" in text:
                hint = " A random_state only matters when the rows are shuffled."
            elif "feature names should match" in text:
                hint = (" It predicts from other columns than the ones it was fitted on. Fit and "
                        "predict with the same df[features].")
            elif isinstance(err, NameError):
                hint = " It uses a name this notebook does not define."
            else:
                hint = ""
            raise AssertionError(f"{where}, kfold_mse stopped with {type(err).__name__}: "
                                 f"{text[:160].rstrip('.')}.{hint}") from None
        assert got is not None, "kfold_mse gives back nothing: use return, not print."
        assert not hasattr(got, "predict"), (
            "kfold_mse returns the fitted model. Return the average of the fold errors.")
        assert np.size(got) == 1, (
            f"kfold_mse returns {np.size(got)} numbers. Return their average, one number.")
        value = float(np.asarray(got, dtype=float).ravel()[0])
        assert np.isfinite(value), f"{where}, kfold_mse gives {value}."
        return value

    used = names_used(kfold_mse.__code__)
    assert "cv_error" not in used, (
        "kfold_mse calls cv_error, which fits a polynomial in one column. Fit the model it is given.")
    assert not used & {"cross_val_score", "cross_validate"}, (
        "kfold_mse hands the whole job to scikit-learn's cross-validation. Write the loop over the "
        "folds yourself.")
    # inspect.signature lists the names of a function's arguments, in order
    params = list(inspect.signature(kfold_mse).parameters)
    assert params[:5] == ["model", "df", "features", "outcome", "k"], (
        f"kfold_mse's arguments are ({', '.join(params)}). The check calls it as the task asks: "
        "kfold_mse(model, df, features, outcome, k).")
    assert len(poly5) == 80 and powers == ["x1", "x2", "x3", "x4", "x5"], (
        "poly5 or powers is no longer the table the GIVEN cell built. Give the names in your cell "
        "other names, then rerun the notebook from the top.")

    # Diagnosis 1's 80 points, degree 5, 5 folds
    got5 = run("On poly5 with 5 folds", LinearRegression(), poly5, powers, "y", 5)
    want5 = float(cvs[4])
    assert same(run("On poly5 a second time", LinearRegression(), poly5, powers, "y", 5), got5), (
        "Two calls with the same arguments give different answers, so the folds change every "
        "time. Fix them with random_state=RANDOM_STATE, as cv_error does.")
    slips = {
        0.16225974927: "scores each fold's model on the rows it was fitted to. Score it on the "
                       "rows of the fold that was left out",
        0.16437573420: "is the training error printed above: the model was fitted on all 80 rows, "
                       "so every fold's rows were in the fit. Fit it inside the loop, on the other "
                       "folds only",
        0.43042838754: "is a square root. Return the mean squared error itself",
        0.42393491148: "averages each fold's root mean squared error. Return the average of the "
                       "fold mean squared errors, with no square root",
        0.34305611672: "averages absolute errors. Square the errors",
        0.18063199809: "comes from folds that were not shuffled. Deal the rows as cv_error does",
        0.18827637518: "comes from random_state=0. Use random_state=RANDOM_STATE, as cv_error does",
        0.92634298401: "adds up the five fold errors. Average them",
        2.96429754883: "adds up each fold's squared errors. Take their mean",
        0.18712086531: "is the first fold's error alone. Is the return line inside the for loop? "
                       "Move it out, so that all k folds run first",
        0.15655130355: "is the last fold's error alone: each pass through the loop throws away "
                       "the folds before it. Make the empty list before the loop",
        0.71448791583: "is the average R squared, model.score. Compute the mean squared error",
        0.12242886674: "fits each fold's model on that fold's own rows and scores it there. Fit "
                       "on the other folds, and score on the fold left out",
        3.06551632461: "fits on one fold and scores on the other four: the training and test "
                       "positions are the wrong way round",
        0.00600369185: "squares the mean error of each fold, so errors of opposite signs cancel. "
                       "Square each error first, then take the mean",
        0.16109964788: "is the median of the fold errors. Take their mean",
        0.03705371936: "divides each fold's sum of squared errors by the number of rows in df. "
                       "Divide by the number of rows in that fold: take the mean",
    }
    for value, cause in slips.items():
        assert not np.isclose(got5, value, rtol=1e-6, atol=0), (
            f"On poly5 with 5 folds, kfold_mse gives {got5:.6f}; Diagnosis 1 printed "
            f"{want5:.4f} for degree 5. Yours {cause}.")
    assert not any(got5 == round(want5, places) for places in range(7)), (
        f"kfold_mse gives {got5}, and the full number is {want5}. Return it unrounded; round "
        "only when you print.")
    if not same(got5, want5):
        # Leave-one-out: with one row per fold, every way of dealing the rows gives the same folds
        loo = run("On poly5 with one row per fold", LinearRegression(), poly5, powers, "y", len(poly5))
        assert not same(loo, float(cv_error(x_tr, y_tr, 5, k=len(poly5)))), (
            f"On poly5 with 5 folds, kfold_mse gives {got5:.6f}; Diagnosis 1 printed {want5:.4f}. "
            "Fitting and scoring are right; only the dealing of rows into folds differs. Use "
            "KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE), as cv_error does.")
    assert same(got5, want5), (
        f"On poly5 with 5 folds, kfold_mse gives {got5:.6f}; Diagnosis 1 printed {want5:.4f} for "
        f"degree 5 ({want5:.8f} unrounded). Compare each step with the task above: how the "
        "rows are dealt into folds, what the model is fitted on, and what it is scored on.")
    assert np.allclose(ols.coef_, LinearRegression().fit(poly5[powers], poly5["y"]).coef_), (
        "The notebook's ols is no longer the fit to all 80 rows: kfold_mse refits ols. Fit the "
        "model it is given, then rerun the poly5 cell.")

    # 10 folds
    got10 = run("On poly5 with 10 folds", LinearRegression(), poly5, powers, "y", 10)
    assert not same(got10, got5), (
        "kfold_mse gives the same answer with 10 folds as with 5. Does it use its k argument?")
    want10 = float(cv_error(x_tr, y_tr, 5, k=10))
    assert same(got10, want10), (
        f"With 10 folds kfold_mse gives {got10:.6f}; it should give {want10:.6f}.")

    # A model that predicts the mean of its training rows
    got_mean = run("With DummyRegressor(), which predicts the training mean", DummyRegressor(),
                   poly5, powers, "y", 5)
    assert not same(got_mean, got5), (
        "With DummyRegressor(), a model that predicts the mean of its training rows, kfold_mse "
        "gives the same answer as with LinearRegression(). Does it fit the model it is given, or "
        "ols or a LinearRegression() of its own?")
    want_mean = float(cv_error(x_tr, y_tr, 0))
    assert same(got_mean, want_mean), (
        f"With DummyRegressor(), kfold_mse gives {got_mean:.6f}; it should give {want_mean:.6f}.")

    # A new sample of 150 rows: other column names and order, a column the model must not use,
    # and row labels that are not 0 to 149
    x_new, y_new = sample(150, 2026)
    renamed = pd.DataFrame({"enrol": y_new, "need": x_new, "need_sq": x_new ** 2,
                            "need_cu": x_new ** 3, "region": x_new ** 4}, index=1000 + 2 * np.arange(150))
    got_new = run("On a new sample with columns enrol, need, need_sq, need_cu and region",
                  LinearRegression(),
                  renamed, ["need", "need_sq", "need_cu"], "enrol", 10)
    assert not same(got_new, got10), (
        "On a new sample of 150 rows, kfold_mse gives its answer for poly5. Does it use its df "
        "argument, or poly5?")
    want_new = float(cv_error(x_new, y_new, 3, k=10))
    # region is need to the fourth power, so fitting on every column but enrol is degree 4
    assert not same(got_new, float(cv_error(x_new, y_new, 4, k=10))), (
        "On the new sample, kfold_mse fits on every column except the outcome, region included. "
        "Fit on the columns named in features only.")
    assert same(got_new, want_new), (
        f"On the new sample, kfold_mse gives {got_new:.6f}; it should give {want_new:.6f}. Does it "
        "use its df, features and outcome arguments everywhere, the split included?")

    print(f"Passed. kfold_mse gives Diagnosis 1's {want5:.4f} for degree 5, and works with other "
          "folds, models and column names.")
    print(f"Degree 5 with 10 folds, from your function: {got10:.4f}   (5 folds: {got5:.4f})")

---

## Ship It: `src/model_selection.py` (10 min)

Here is the finished module. Its design rule is this lab's: a function that *chooses* a model
returns a nested score with its choice, so the number it hands back was never used to make the
choice. `cv_score` is the plain k-fold score, honest only for a degree fixed in advance, and
`select` chooses a degree and scores the choice by nested CV, as in Diagnosis 2.

This part is reading only: run the two cells and compare the output with the expected output below.
The first saves the module as a file, the second imports it and runs it on the analyst's data.

**Reading the cells.** `%%writefile src/model_selection.py` saves the cell as that file instead of
running it. In `select`, `choose` is a small function defined inside another, used twice: once
inside each outer fold, and once on all the data for the degree you deploy. `select` returns a
dictionary, read with `result["choice"]`.

In [ ]:
%%writefile src/model_selection.py
"""Model selection whose reported score was never used to make the choice (ECON 5200 Lab 15).

The design rule: a function that chooses a model returns a nested score with its choice, never
the winning CV score.
"""
import numpy as np
from sklearn.model_selection import KFold


def cv_score(x, y, deg, k=5, seed=42):
    """Plain k-fold MSE of a degree-deg polynomial. Honest only for a degree fixed in advance."""
    errors = []
    for train, test in KFold(n_splits=k, shuffle=True, random_state=seed).split(x):
        coefs = np.polyfit(x[train], y[train], deg)
        errors.append(np.mean((np.polyval(coefs, x[test]) - y[test]) ** 2))
    return float(np.mean(errors))


def select(x, y, degrees, k=5, seed=42):
    """Choose a degree by k-fold CV, and score that choice by nested CV."""
    def choose(x_in, y_in):
        scores = [cv_score(x_in, y_in, d, k, seed) for d in degrees]
        return degrees[int(np.argmin(scores))]

    nested = []
    for train, test in KFold(n_splits=k, shuffle=True, random_state=seed).split(x):
        best = choose(x[train], y[train])
        coefs = np.polyfit(x[train], y[train], best)
        nested.append(np.mean((np.polyval(coefs, x[test]) - y[test]) ** 2))
    return {"choice": choose(x, y), "nested_score": float(np.mean(nested))}

In [ ]:
# GIVEN — run as-is
# Import the module and run it on the analyst's data: the numbers should match Diagnoses 1 and 2
import importlib
import sys

sys.path.insert(0, "src")              # let Python find modules in the src folder
import model_selection
importlib.reload(model_selection)      # reload, so a re-run of the %%writefile cell takes effect

result = model_selection.select(x_tr, y_tr, list(degs))
print(f"chosen degree {result['choice']}, nested CV score {result['nested_score']:.4f}")
print(f"plain CV score of degree 5: {model_selection.cv_score(x_tr, y_tr, 5):.4f}")

**Expected output:** chosen degree 5 with a nested CV score of 0.1910, as in Diagnosis 2, and a
plain CV score of 0.1853 for degree 5, as in Diagnosis 1.

---

## Extension (Optional — 20 min): One Standard Error

Instead of the degree with the smallest CV error, the one-standard-error rule chooses the
*simplest* degree whose CV error is within one standard error of that minimum. The standard error
of a degree's CV error is the standard deviation of its five fold errors (`ddof=1`) divided by
√5.

**The task.** In the cell below, compute each degree's five fold errors on the analyst's folds,
apply the rule, and compare the degree it chooses, and that fit's error on the 20,000 new points,
with the analyst's degree 5 and its 0.2330.

**Your answer.** Does the rule help on this sample? Explain what it guards against, in terms of
the optimism you measured in Diagnosis 1.

In [ ]:
# EXTENSION (optional) — the one-standard-error rule

*Your answer here:*

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Cross-validated polynomials of degree 1 to 12 on 80 simulated
  points, chose degree [YOUR VALUE], and its CV error was [YOUR VALUE]
* Scored that fit on 20,000 new points from the same simulation:
  its true error was [YOUR VALUE]
* Ran nested cross-validation, which estimated [YOUR VALUE]
* Split the expected error into bias squared, variance and noise
  by simulation; the noise alone was [YOUR VALUE]
* Wrote my own k-fold cross-validation function for scikit-learn
  models and checked it against the lab: with 10 folds, degree 5
  gave [YOUR VALUE]

**Please write a README.md entry including:**
1. Project Title: Bias, Variance and an Honest Cross-Validation Score
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust',
'comprehensive' or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab15-bias-variance`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab15-bias-variance`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab15-bias-variance`, branch `main`,
   commit message `Lab 15 submission`. This pushes the notebook, and the
   notebook's rendered output is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/model_selection.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.